### Import Libraries

In [1]:
import pandas as pd
!pip install openai
import os

In [2]:
import re
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer

from transformers import pipeline

### Load dataset

In [3]:
df = pd.read_csv(r"C:\Users\Shrabani P\IronHack\Week6_Day2\Project_3\Project3_amazon-review-insights\data\amazon_reviews_clustered.csv")

In [4]:
df.head()

,id,name,brand,categories,asins,reviews.rating,reviews.title,reviews.text,reviews.username,reviews.date,review_length,cluster_text,Cluster,Meta_Category
0,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...",B01AHB9CN2,5.0,Kindle,This product so far has not disappointed. My c...,Adapter,2017-01-13 00:00:00+00:00,27,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",2,Tablets & eReaders
1,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...",B01AHB9CN2,5.0,very fast,great for beginner or experienced person. Boug...,truman,2017-01-13 00:00:00+00:00,14,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",2,Tablets & eReaders
2,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...",B01AHB9CN2,5.0,Beginner tablet for our 9 year old son.,Inexpensive tablet for him to use and learn on...,DaveZ,2017-01-13 00:00:00+00:00,26,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",2,Tablets & eReaders
3,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...",B01AHB9CN2,4.0,Good!!!,I've had my Fire HD 8 two weeks now and I love...,Shacks,2017-01-13 00:00:00+00:00,117,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",2,Tablets & eReaders
4,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...",B01AHB9CN2,5.0,Fantastic Tablet for kids,I bought this for my grand daughter when she c...,explore42,2017-01-12 00:00:00+00:00,117,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",2,Tablets & eReaders


### Find the Top 3 Products in Each Category

In [5]:
product_stats = (
    df.groupby(["Meta_Category", "name"])
      .agg(
          avg_rating=("reviews.rating", "mean"),
          review_count=("reviews.rating", "count")
      )
      .reset_index()
)

### Keep products with enough reviews

In [6]:
product_stats = product_stats[
    product_stats["review_count"] >= 20
]

### Select Top 3 Products

In [7]:
top_products = (
    product_stats
    .sort_values(
        ["Meta_Category", "avg_rating"],
        ascending=[True, False]
    )
    .groupby("Meta_Category")
    .head(3)
)

top_products

,Meta_Category,name,avg_rating,review_count
0,Batteries & Household Essentials,AmazonBasics AA Performance Alkaline Batteries...,4.414875,3442
1,Batteries & Household Essentials,AmazonBasics AAA Performance Alkaline Batterie...,4.397141,7554
5,Electronics & Streaming Devices,Unknown Product,4.707278,5056
16,Smart Home & Alexa Devices,"Amazon Fire Hd 10 Tablet, Wi-Fi, 16 Gb, Specia...",4.773438,128
8,Smart Home & Alexa Devices,Amazon - Echo Plus w/ Built-In Hub - Silver,4.748727,589
7,Smart Home & Alexa Devices,Amazon - Amazon Tap Portable Bluetooth and Wi-...,4.729560,318
68,Tablets & eReaders,Amazon Kindle Paperwhite - eBook reader - 4 GB...,4.755038,3176
123,Tablets & eReaders,"Kindle Voyage E-reader, 6 High-Resolution Disp...",4.743103,580
61,Tablets & eReaders,Amazon Fire Hd 8 8in Tablet 16gb Black B018szt...,4.740741,135


### Collect Reviews of Those Products

In [8]:
selected_reviews = df.merge(
    top_products[
        ["Meta_Category","name"]
    ],
    on=["Meta_Category","name"]
)

In [9]:
print("Total reviews:", len(selected_reviews))

Total reviews: 20978


### Split Reviews into Sentences

In [10]:
def split_sentences(text):

    text = str(text)

    sentences = re.split(
        r'(?<=[.!?])\s+',
        text
    )

    return [
        s.strip()
        for s in sentences
        if 15 < len(s.strip()) < 300
    ]

### TF-IDF Sentence Selection

In [11]:
def top_representative_sentences(texts, n=40):

    all_sentences = []

    for text in texts:

        all_sentences.extend(
            split_sentences(text)
        )

    if len(all_sentences)==0:
        return []

    # Remove duplicate sentences

    all_sentences = list(
        dict.fromkeys(all_sentences)
    )

    vectorizer = TfidfVectorizer(
        stop_words="english",
        max_features=2000
    )

    X = vectorizer.fit_transform(all_sentences)

    scores = np.asarray(
        X.sum(axis=1)
    ).ravel()

    top_idx = scores.argsort()[::-1][:n]

    return [
        all_sentences[i]
        for i in sorted(top_idx)
    ]

In [12]:
print(selected_reviews.head())

                     id                                               name  \
0  AVqVGWLKnnc1JgDc3jF1  Amazon Kindle Paperwhite - eBook reader - 4 GB...   
1  AVqVGWLKnnc1JgDc3jF1  Amazon Kindle Paperwhite - eBook reader - 4 GB...   
2  AVqVGWLKnnc1JgDc3jF1  Amazon Kindle Paperwhite - eBook reader - 4 GB...   
3  AVqVGWLKnnc1JgDc3jF1  Amazon Kindle Paperwhite - eBook reader - 4 GB...   
4  AVqVGWLKnnc1JgDc3jF1  Amazon Kindle Paperwhite - eBook reader - 4 GB...   

    brand                                         categories       asins  \
0  Amazon  Tablets,Fire Tablets,Computers & Tablets,All T...  B018Y23MNM   
1  Amazon  Tablets,Fire Tablets,Computers & Tablets,All T...  B018Y23MNM   
2  Amazon  Tablets,Fire Tablets,Computers & Tablets,All T...  B018Y23MNM   
3  Amazon  Tablets,Fire Tablets,Computers & Tablets,All T...  B018Y23MNM   
4  Amazon  Tablets,Fire Tablets,Computers & Tablets,All T...  B018Y23MNM   

   reviews.rating          reviews.title  \
0             3.0      I like 

In [13]:
category = "Tablets & eReaders"

texts = selected_reviews.loc[
    selected_reviews["Meta_Category"] == category,
    "reviews.text"
]

### Create the input text

In [14]:
selected_sentences = top_representative_sentences(texts, n=40)

### Set API Key

In [15]:
import os

print(os.getenv("OPENAI_API_KEY"))

None


In [16]:
import os

print(os.getcwd())

c:\Users\Shrabani P\IronHack\Week6_Day2\Project_3\Project3_amazon-review-insights\notebooks\05_Summarization\OpenAI_API


In [17]:
import os

print(os.listdir())

['.env', '5_3_openai_api.ipynb', 'openai_summaries.csv', 'openai_summaries.pkl', 'Save_model']


In [18]:
from dotenv import load_dotenv
import os
from openai import OpenAI

load_dotenv(".env")

api_key = os.getenv("OPENAI_API_KEY")

print(api_key[:15] if api_key else "No API key found")

client = OpenAI(api_key=api_key)

sk-proj-4rG1YW7


In [19]:
print(client)

### Create a list to store the summaries

### Generate summaries for each category

In [20]:
openai_results = []

for category in selected_reviews["Meta_Category"].unique():

    print(f"Processing: {category}")

    texts = selected_reviews.loc[
        selected_reviews["Meta_Category"] == category,
        "reviews.text"
    ]

    # Get representative sentences
    selected_sentences = top_representative_sentences(texts, n=40)

    input_text = " ".join(selected_sentences)

    if not api_key or not api_key.startswith("sk-"):
        raise ValueError("OPENAI_API_KEY is missing or invalid. Set a valid key before running this cell.")

    try:
        response = client.chat.completions.create(
            model="gpt-4.1",
            temperature=0.3,
            messages=[
                {
                    "role": "system",
                    "content": """
You are an expert product review analyst.

Generate clear, objective, and well-structured summaries from customer reviews.
Base your response only on the provided reviews.
Do not invent products or features that are not mentioned.
"""
                },
                {
                    "role": "user",
                    "content": f"""
Summarize the following customer reviews for the product category: {category}.

Your summary should include:

1. Top 3 products and their key strengths.
2. Key differences between the products.
3. Most common customer complaints for each product.
4. The product with the most negative feedback and why.
5. Overall buying recommendation.

Customer Reviews:

{input_text}
"""
                }
            ]
        )
    except Exception as e:
        print(f"OpenAI API error for '{category}': {e}")
        continue

    summary = response.choices[0].message.content

    openai_results.append({
        "Meta_Category": category,
        "OpenAI_Summary": summary
    })



Processing: Tablets & eReaders
Processing: Smart Home & Alexa Devices
Processing: Electronics & Streaming Devices
Processing: Batteries & Household Essentials


In [21]:
summary = response.choices[0].message.content

openai_results.append({
    "Meta_Category": category,
    "OpenAI_Summary": summary
})

print(f"✓ Completed: {category}")

✓ Completed: Batteries & Household Essentials


In [22]:
print(type(openai_results))
print(len(openai_results))
print(openai_results)

<class 'list'>
5
[{'Meta_Category': 'Tablets & eReaders', 'OpenAI_Summary': '**Tablets & eReaders Customer Review Summary**\n\n---\n\n### 1. **Top 3 Products and Their Key Strengths**\n\n**a. Kindle Paperwhite**\n- **Strengths:** \n  - Lightweight and easy to hold\n  - Excellent battery life (2-8 weeks depending on use)\n  - High-resolution e-ink screen, crisp text, and adjustable backlight for comfortable reading in all lighting conditions\n  - Large selection of books, often at lower prices than competitors\n  - Features like Vocabulary Builder and adjustable font sizes/styles\n  - Reliable and durable; suitable for both indoor and outdoor reading\n\n**b. Kindle Voyage**\n- **Strengths:**\n  - Auto-brightness feature for self-adjusting backlight\n  - Pressure-sensitive page turn buttons and responsive touchscreen\n  - Lightweight and portable\n  - High-quality display with no glare in sunlight\n  - Additional features such as quick dictionary lookup, note-taking, and easy book transf

### Save the summaries

In [24]:
openai_results = pd.DataFrame(openai_results)

openai_results.to_csv("openai_summaries.csv", index=False)

openai_results

,Meta_Category,OpenAI_Summary
0,Tablets & eReaders,**Tablets & eReaders Customer Review Summary**...
1,Smart Home & Alexa Devices,**Smart Home & Alexa Devices: Customer Review ...
2,Electronics & Streaming Devices,**Electronics & Streaming Devices – Customer R...
3,Batteries & Household Essentials,**Summary of Customer Reviews: Batteries & Hou...
4,Batteries & Household Essentials,**Summary of Customer Reviews: Batteries & Hou...


In [25]:
openai_results.to_pickle("openai_summaries.pkl")